In [78]:
import pandas as pd
import evaluation as ev
import re

### Explore the DataFrame

In [2]:
df = pd.read_csv("entailment_probs_or.csv")

In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,pairID,gold_label,Sentence1,Sentence2,Explanation_1,Sentence1_marked_1,Sentence2_marked_1,Explanation_2,...,Sentence1_marked_3,Sentence2_marked_3,Sentence1_Highlighted_Ordered_1,Sentence2_Highlighted_Ordered_1,Sentence1_Highlighted_Ordered_2,Sentence2_Highlighted_Ordered_2,Sentence1_Highlighted_Ordered_3,Sentence2_Highlighted_Ordered_3,explanation_type,matching_explanations
0,1,1,4705552913.jpg#2r1e,entailment,Two women are embracing while holding to go pa...,Two woman are holding packages.,Saying the two women are holding packages is a...,Two women are embracing while holding *to* *g...,Two woman are *holding* *packages.*,Sentence 1 states that two women are holding t...,...,Two *women* are *embracing* while holding to ...,Two woman are *holding* *packages.*,"['to', 'go', 'packages.']","['holding', 'packages.']","['Two', 'women', 'holding', 'packages.']",[],"['women', 'embracing']","['holding', 'packages.']",classification,2
1,33,33,3021028400.jpg#1r1e,entailment,"Man in a black suit, white shirt and black bow...",A person in a suit,"A man is a person, and both sentences mention ...","*Man* in a black *suit,* white shirt and blac...",A *person* in a *suit*,"Man is person, and black suit is a type of suit.",...,"*Man* in a black suit, white shirt and black ...",A *person* in a suit,"['Man', 'suit']","['person', 'suit']","['Man', 'black', 'suit']","['person', 'suit']",['Man'],['person'],classification,"2,3"
2,65,65,1257692349.jpg#2r1e,entailment,A group of onlookers glance at a person doing ...,People watch another person do a trick.,People watch a person doing a strange trick.,A group of onlookers glance at a *person* *do...,*People* *watch* another person do a *trick.*,"People watch another person ""do a trick"" and t...",...,A *group* *of* *onlookers* glance at a person...,*People* watch another person do a trick.,"['person', 'doing', 'strange']","['People', 'watch', 'trick.']","['strange', 'trick']",['trick.'],"['group', 'of', 'onlookers']",['People'],classification,3
3,84,84,8162775105.jpg#3r1e,entailment,A pregnant lady singing on stage while holding...,A woman is making music.,Pregnant lady implies woman.,A *pregnant* *lady* singing on stage while ho...,A *woman* is making music.,"Pregnant lady is a type of woman, and singing ...",...,*A* *pregnant* *lady* *singing* on stage whil...,*A* *woman* *is* *making* *music.*,"['pregnant', 'lady']",['woman'],"['pregnant', 'lady', 'singing']","['woman', 'making', 'music.']","['A', 'pregnant', 'lady', 'singing']","['A', 'woman', 'is', 'making', 'music.']",classification,"2,3"
4,86,86,5777129645.jpg#2r1n,entailment,The two farmers are working on a piece of John...,Men are working on John Deere equipment,The two farmers can also be denoted as men as ...,The *two* *farmers* are working on a piece of...,*Men* are *working* on John Deere *equipment*,"Often farmers are men, and they are working on...",...,The two *farmers* are working on a piece of J...,*Men* are working on John Deere equipment,"['two', 'farmers']","['Men', 'working', 'equipment']","['working', 'John', 'Deere', 'equipment.']","['working', 'John', 'Deere', 'equipment']",['farmers'],['Men'],classification,3


In [5]:
def print_example(df, ID = None, rownum = None):
    """
        Function to print a specific example from the DataFrame
        param: df (pd.Dataframe): dataframe containing a pairID column
        param: ID (str): ID to be considered
        param: rownum (int): row number one wants to print
        return: None
    """
    
    if ID is not None: 
        row = df.loc[df["pairID"] == ID]
        if row.empty:
            print("ID not found.")
            return
        row = row.iloc[0]  
    if rownum is not None:
        row = df.iloc[rownum]
    for col in df.columns:
        #if "Highlighted" in col: 
            #continue
        #else: 
        print(f"{col}:")
        print(row[col])
        print()

In [6]:
#Choose which dataframe to use by uncommenting the lines below

df = pd.read_csv("entailment_probs_or.csv")
#df = pd.read_csv("processed_esnli_EA.csv")
#df = pd.read_csv("esnli_dev.csv")

#change this number to choose how many examples to print
n = 1
print(f"amount of rows:{df.shape[0]}")        
for i in range(0,n):
    print(f"----------------EXAMPLE {i + 1} ----------------\n")
    print_example(df, rownum=i)


amount of rows:576
----------------EXAMPLE 0 ----------------

Unnamed: 0.1:
1

Unnamed: 0:
1

pairID:
4705552913.jpg#2r1e

gold_label:
entailment

Sentence1:
Two women are embracing while holding to go packages.

Sentence2:
Two woman are holding packages.

Explanation_1:
Saying the two women are holding packages is a way to paraphrase that the packages they are holding are to go packages.

Sentence1_marked_1:
 Two women are embracing while holding *to* *go* *packages.*

Sentence2_marked_1:
 Two woman are *holding* *packages.*

Explanation_2:
Sentence 1 states that two women are holding to-go packages. To-go packages are a form of package.

Sentence1_marked_2:
 *Two* *women* are embracing while *holding* to go *packages.*

Sentence2_marked_2:
 Two woman are holding packages.

Explanation_3:
Women can embrace while they are holding packages.

Sentence1_marked_3:
 Two *women* are *embracing* while holding to go packages.

Sentence2_marked_3:
 Two woman are *holding* *packages.*

Sentence

### Answer Extraction from annotators

In [80]:
# Change pid to get different examples
pid = "3089862485.jpg#0r1e"

one_df = df[df["pairID"] == pid].reset_index()
one_df

,index,Unnamed: 0.1,Unnamed: 0,pairID,gold_label,Sentence1,Sentence2,Explanation_1,Sentence1_marked_1,Sentence2_marked_1,...,Sentence1_marked_3,Sentence2_marked_3,Sentence1_Highlighted_Ordered_1,Sentence2_Highlighted_Ordered_1,Sentence1_Highlighted_Ordered_2,Sentence2_Highlighted_Ordered_2,Sentence1_Highlighted_Ordered_3,Sentence2_Highlighted_Ordered_3,explanation_type,matching_explanations
0,521,8961,8961,3089862485.jpg#0r1e,entailment,A man on stage holds a hello kitty guitar and ...,The man is holding an instrument in his hands.,A guitar is a type of instrument.,A man on stage holds a hello kitty *guitar* a...,The man is holding an *instrument* in his hands.,...,A *man* *on* *stage* holds a *hello* *kitty* ...,The *man* is holding an *instrument* in his h...,['guitar'],['instrument'],"['hello', 'kitty', 'guitar']",['instrument'],"['man', 'on', 'stage', 'hello', 'kitty', 'guit...","['man', 'instrument']",classification,"1,3"


In [81]:
print(f"Premise: {one_df['Sentence1'][0]}")
print(f"Hypothesis: {one_df['Sentence2'][0]}")
print(f"Explanation 1: {one_df['Explanation_1'][0]}")
print(f"Explanation 2: {one_df['Explanation_2'][0]}")
print(f"Explanation 3: {one_df['Explanation_3'][0]}")
print("-"*20)

Premise: A man on stage holds a hello kitty guitar and looks to the right of the picture.
Hypothesis: The man is holding an instrument in his hands.
Explanation 1: A guitar is a type of instrument.
Explanation 2: guitar is a musical instrument, which is always held in hands
Explanation 3: Man on stage is a type of man, and hello kitty guitar is a type of instrument.
--------------------


In [79]:
#Showcase of how the get_overlap function extracts the highlights from the explanations

row = one_df.iloc[0]
ann_matches = row["matching_explanations"].split(",")

for ann_i in ann_matches:
    print("\nAnnotator:", ann_i)

    splitted_ex = re.split(r"(?:type of|form of|kind of)",
                           row[f"Explanation_{ann_i}"])

    for j in range(len(splitted_ex) - 1):
        left_window = splitted_ex[j].split()[-6:]
        right_window = splitted_ex[j + 1].split()[:4]

        left_overlap = ev.get_overlap(
            left_window,
            row[f"Sentence1_Highlighted_Ordered_{ann_i}"]
        )

        right_overlap = ev.get_overlap(
            right_window,
            row[f"Sentence2_Highlighted_Ordered_{ann_i}"]
        )

        print("Left overlap:", left_overlap)
        print("Right overlap:", right_overlap)


Annotator: 1
Left overlap: ['guitar']
Right overlap: ['instrument']

Annotator: 3
Left overlap: ['man', 'on', 'stage']
Right overlap: ['man']
Left overlap: ['hello', 'kitty', 'guitar']
Right overlap: ['instrument']


In [83]:
# Construction of extracted format
answers_dict, missing = ev.get_correct_answers(one_df)

print("Missing:", missing)
print("\nExtracted structure:")
print(answers_dict[pid])

Missing: set()

Extracted structure:
[{'left': [['guitar'], ['hello', 'kitty', 'guitar']], 'right': [['instrument'], ['instrument']], 'annotator_nr': ['1', '3']}, {'left': [['man', 'on', 'stage']], 'right': [['man']], 'annotator_nr': ['3']}]


### Analysis of one example when computing metrics

In [14]:
import evaluation as ev

responses_LLM = ev.read_json('final_LLM_auto_responses_gemini-2.5-flash.json')
example = responses_LLM['q0952']
example

{'answer': 'entailment',
 'explanation': ['a hello kitty guitar is a type of an instrument']}

In [13]:
answers_extracted = ev.read_json('annotators_answers.json')
ans = answers_extracted['3089862485.jpg#0r1e']
ans

[{'left': [['guitar'], ['hello', 'kitty', 'guitar']],
  'right': [['instrument'], ['instrument']],
  'annotator_nr': ['1', '3']},
 {'left': [['man', 'on', 'stage']], 'right': [['man']], 'annotator_nr': ['3']}]

In [15]:
pairID = '3089862485.jpg#0r1e'

llm_ex = {pairID: responses_LLM['q0952']}   
gold_ex = {pairID: answers_extracted['3089862485.jpg#0r1e']}

print("LLM raw explanations:")
print(llm_ex[pairID].get("explanation", []))
print("\nGold answer groups:")
print(gold_ex[pairID])


LLM raw explanations:
['a hello kitty guitar is a type of an instrument']

Gold answer groups:
[{'left': [['guitar'], ['hello', 'kitty', 'guitar']], 'right': [['instrument'], ['instrument']], 'annotator_nr': ['1', '3']}, {'left': [['man', 'on', 'stage']], 'right': [['man']], 'annotator_nr': ['3']}]


In [40]:
import importlib
import evaluation

importlib.reload(ev)
ev.check_LLM_answer(gold_ex, llm_ex, max_extra_total=2, allow_implies=True, verbose = True)


PAIR: 3089862485.jpg#0r1e
Raw explanations: ['a hello kitty guitar is a type of an instrument']
Filtered relations: ['a hello kitty guitar is a type of an instrument']
Predicted label: entailment

--- LLM relation 1 ---
Relation: a hello kitty guitar is a type of an instrument
Tokenized llm_left : ['hello', 'kitty', 'guitar']
Tokenized llm_right: ['instrument']
Pred length: 4

  Gold group 1: {'left': [['guitar'], ['hello', 'kitty', 'guitar']], 'right': [['instrument'], ['instrument']], 'annotator_nr': ['1', '3']}
    left_exact/right_exact: True True
    left_partial/right_partial: True True
    left_extras_candidates: [2, 0]
    right_extras_candidates: [0, 0]
    -> FOUND EXACT in this group
=> counts: exact +1, len_ok +1, partial +1

PAIR RESULT: {'exact': 1, 'partial': 1, 'len_ok': 1, 'combined_correct': 1, 'combined_len_ok': 1, 'total_answers': 2, 'total_LLM_answers': 1, 'avg_len': 4.0, 'gold_avg_len': 3.2, 'len_ratio': 1.25}

Pairs with no usable relation: 0
Pairs with empty/no

({'3089862485.jpg#0r1e': {'exact': 1,
   'partial': 1,
   'len_ok': 1,
   'combined_correct': 1,
   'combined_len_ok': 1,
   'total_answers': 2,
   'total_LLM_answers': 1,
   'avg_len': 4.0,
   'gold_avg_len': 3.2,
   'len_ratio': 1.25}},
 {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666},
 {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666},
 {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666},
 {'TP': 1, 'FP': 0, 'FN': 1},
 0,
 0,
 4.0,
 3.2)